## stats follows by plotting

In [ ]:
import pandas as pd
df=pd.read_csv("melted_biofilm_for_fig1.csv") # or biofilm_polystyrene_plate.csv

In [ ]:
# df

In [ ]:
# final stats for paper

## --- Statistical analysis ---
## 1. Technical replicates (n=3) are averaged within each biological replicate
##    to avoid pseudoreplication — true sample size is n=3 (biological reps).
## 2. Randomized block ANOVA: tests for strain effect while blocking by
##    experimental day (replicate), since all strains were processed together
##    on each day — this removes day-to-day noise.
## 3. Tukey HSD post-hoc: all pairwise comparisons (to be reported in the supp excel sheet)

import pandas as pd
from statsmodels.formula.api import ols
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Filter and average
bio = df[df['expt'] == 'Biofilm']
bio_means = bio.groupby(['strain', 'replicate'])['value'].mean().reset_index()

# Randomized block ANOVA (strain effect + day block)
model = ols('value ~ C(strain) + C(replicate)', data=bio_means).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print("--- ANOVA (Biofilm Only) ---")
print(anova_table)

# Tukey HSD post-hoc (all pairwise)
posthoc = pairwise_tukeyhsd(endog=bio_means['value'], groups=bio_means['strain'], alpha=0.05)
print("\n--- Tukey HSD (all pairwise) ---")
print(posthoc)

Final figure for USK paper 2026

In [ ]:
import pandas as pd
df=pd.read_csv("melted_biofilm_for_fig1.csv") # or biofilm_polystyrene_plate.csv 

In [ ]:
# final figure for paper
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Panel / style (compact, white bg, y-grid only)
# -----------------------------
sns.set(style="darkgrid", font_scale=1.2)

fig, ay = plt.subplots(figsize=(3, 3), dpi=600)

ay.grid(False)
ay.grid(axis="y", linestyle="-", linewidth=0.8)

for spine in ay.spines.values():
    spine.set_edgecolor("black")
    spine.set_linewidth(1.0)

# -----------------------------
# Data prep — biofilm only
# -----------------------------
df_bio = df[df["expt"] == "Biofilm"].copy()

order = list(df_bio["strain"].unique())
positions = np.arange(len(order)) + 1.0
width = 0.85

vals = [df_bio.loc[df_bio["strain"] == s, "value"].dropna().to_numpy()
        for s in order]

# -----------------------------
# Boxplot (matplotlib) — gainsboro fill, like swarm plot
# -----------------------------
bp = ay.boxplot(
    vals,
    positions=positions,
    widths=width,
    patch_artist=True,
    showfliers=False,
    showcaps=False,
    whiskerprops=dict(linewidth=1.2, color="dimgray"),
    medianprops=dict(linewidth=1.4, color="dimgray"),
    boxprops=dict(linewidth=1.2, color="dimgray", facecolor= "gainsboro"),
)

# -----------------------------
# Overlay individual points (solid, rebeccapurple)
# -----------------------------
rng = np.random.default_rng(7)
for x, y in zip(positions, vals):
    jitter = rng.normal(0, 0.04, size=len(y))
    ay.scatter(
        np.full(len(y), x) + jitter,
        y,
        s=43,
        color="rebeccapurple",
        alpha=0.6,
        zorder=3,
    )

# -----------------------------
# Axes formatting (compact panel)
# -----------------------------
ay.set_xlabel("Strain")
ay.set_ylabel("Abs590")
ay.set_ylim(-0.03, 0.62)
ay.set_yticks([0.2, 0.4, 0.6])

ay.set_xticks(positions)
ay.set_xticklabels(order, rotation=0)
ay.set_xlim(positions[0] - 0.5, positions[-1] + 0.5)

ay.margins(x=0.01)
plt.tight_layout(pad=0.25)

plt.savefig("biofilm_panel2.png", dpi=600, bbox_inches="tight")
plt.show()